# 🏥 Análisis Exploratorio — BI Justicia en Salud Guatemala (IGSS)

**Objetivo:** Exploración inicial de los datos del IGSS para identificar patrones,
anomalías y formular hipótesis sobre corrupción en el sector salud.

**Fuente:** IGSS en Cifras 2025 — Departamento Actuarial y Estadístico

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
COLORS = {'normal': '#27AE60', 'warning': '#E67E22', 'danger': '#C0392B', 'primary': '#1A3A5C'}

# Ruta al Data Warehouse
DB_PATH = Path('../data/warehouse/igss_salud_dw.db')
conn = sqlite3.connect(DB_PATH)
print(f'Conectado a: {DB_PATH}')
print(f'Existe el DW: {DB_PATH.exists()}')

## 1. Exploración del Data Warehouse

In [ ]:
# Verificar conteos en todas las tablas
tablas = ['dim_tiempo', 'dim_departamento', 'dim_servicio', 
          'fact_costos', 'fact_ejecucion', 'red_flags_log']

print('═' * 40)
print('CONTENIDO DEL DATA WAREHOUSE')
print('═' * 40)
for tabla in tablas:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {tabla}', conn).iloc[0,0]
    print(f'  {tabla:<25} {n:>6} registros')
print('═' * 40)

## 2. Análisis de Costos Unitarios Históricos (2014-2024)

**Hipótesis H1:** El incremento de costos de hospitalización en 2020–2021 supera lo que puede explicarse por COVID-19.

In [ ]:
# Cargar datos de costos históricos
df_costos = pd.read_sql_query("""
    SELECT t.anio, s.nombre AS servicio, s.codigo,
           fc.costo_unitario_q, fc.costo_unitario_real,
           fc.z_score, fc.es_outlier, fc.variacion_pct
    FROM fact_costos fc
    JOIN dim_tiempo t   ON fc.id_tiempo = t.id_tiempo
    JOIN dim_servicio s ON fc.id_servicio = s.id_servicio
    WHERE t.es_anual = 1
    ORDER BY s.codigo, t.anio
""", conn)

print('Dimensiones:', df_costos.shape)
df_costos[df_costos['codigo'] == 'HOSP'].describe()

In [ ]:
# ── Gráfica 1: Evolución de costos de hospitalización ────────────────────
df_hosp = df_costos[df_costos['codigo'] == 'HOSP'].copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle('Costo Unitario de Hospitalización IGSS — 2014 a 2024', 
             fontsize=14, fontweight='bold', y=1.01)

# Panel superior: línea de costos
colores_puntos = [COLORS['danger'] if o else COLORS['primary'] for o in df_hosp['es_outlier']]
ax1.plot(df_hosp['anio'], df_hosp['costo_unitario_q'], 
         color=COLORS['primary'], linewidth=2.5, zorder=2)
ax1.scatter(df_hosp['anio'], df_hosp['costo_unitario_q'], 
            c=colores_puntos, s=80, zorder=3)

# Sombrear zona de pandemia
ax1.axvspan(2019.5, 2022.5, alpha=0.08, color='red', label='Período pandemia COVID-19')
ax1.axhline(y=10000, linestyle='--', color=COLORS['warning'], alpha=0.7, label='Umbral alerta (Q10,000)')

# Anotar outliers
for _, row in df_hosp[df_hosp['es_outlier']].iterrows():
    ax1.annotate(f"Q{row['costo_unitario_q']:,.0f}\n(Z={row['z_score']:.1f})",
                 xy=(row['anio'], row['costo_unitario_q']),
                 xytext=(15, 10), textcoords='offset points',
                 fontsize=8, color=COLORS['danger'],
                 arrowprops=dict(arrowstyle='->', color=COLORS['danger'], lw=1.2))

ax1.set_ylabel('Costo Unitario (Q)', fontsize=11)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}'))
ax1.legend(fontsize=9)
ax1.set_title('Costo por egreso hospitalario', fontsize=11)

# Panel inferior: variación porcentual
colores_bar = [COLORS['danger'] if abs(v or 0) > 30 else COLORS['success'] 
               for v in df_hosp['variacion_pct']]
ax2.bar(df_hosp['anio'], df_hosp['variacion_pct'].fillna(0), 
        color=colores_bar, alpha=0.8)
ax2.axhline(y=30, linestyle='--', color=COLORS['warning'], alpha=0.7, label='Umbral 30%')
ax2.axhline(y=-30, linestyle='--', color=COLORS['warning'], alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_ylabel('Variación anual (%)', fontsize=10)
ax2.set_xlabel('Año', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../exports/grafica_costos_historicos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada: exports/grafica_costos_historicos.png')

In [ ]:
# ── Estadísticas del período anómalo ─────────────────────────────────────
pre_pandemia = df_hosp[df_hosp['anio'] <= 2019]['costo_unitario_q'].mean()
pandemia     = df_hosp[df_hosp['anio'].isin([2020, 2021])]['costo_unitario_q'].mean()
post_pandemia = df_hosp[df_hosp['anio'] >= 2022]['costo_unitario_q'].mean()

print('═' * 55)
print('ANÁLISIS DEL PERÍODO DE PANDEMIA (H1)')
print('═' * 55)
print(f'Promedio PRE-pandemia  (2014-2019): Q{pre_pandemia:>10,.2f}')
print(f'Promedio EN pandemia   (2020-2021): Q{pandemia:>10,.2f}  ⚠️')
print(f'Promedio POST-pandemia (2022-2024): Q{post_pandemia:>10,.2f}')
print(f'Incremento pandemia vs pre:         {(pandemia/pre_pandemia - 1)*100:>9.1f}%')
print(f'Post-pandemia vs pre:               {(post_pandemia/pre_pandemia - 1)*100:>9.1f}%')
print('═' * 55)
print('\n⚠️  Los costos post-pandemia NO retornaron a niveles pre-pandemia')
print('    Esto sugiere un cambio estructural en los costos del IGSS.')

## 3. Análisis de Brechas Financieras por Departamento

**Hipótesis H3:** Los departamentos con mayor brecha tienen menor eficiencia operativa.

In [ ]:
# Cargar datos de ejecución financiera
df_ejec = pd.read_sql_query("""
    SELECT t.anio, d.nombre AS departamento, d.region,
           fe.egresos_q, fe.ingresos_q, fe.brecha_q,
           fe.ratio_ei, fe.nivel_riesgo, fe.flag_anomalia
    FROM fact_ejecucion fe
    JOIN dim_tiempo t ON fe.id_tiempo = t.id_tiempo
    JOIN dim_departamento d ON fe.id_departamento = d.id_departamento
    WHERE d.id_departamento != 30
    ORDER BY t.anio, fe.ratio_ei DESC
""", conn)

df_2025 = df_ejec[df_ejec['anio'] == 2025].copy()
print(f'Departamentos analizados 2025: {len(df_2025)}')
print(f'Con flag_anomalia: {df_2025["flag_anomalia"].sum()}')
df_2025[['departamento', 'egresos_q', 'ingresos_q', 'brecha_q', 'ratio_ei', 'nivel_riesgo']].head(10)

In [ ]:
# ── Gráfica 2: Ratio E/I por departamento (2025) ─────────────────────────
nivel_color_map = {
    'NORMAL': COLORS['normal'], 'MODERADO': '#F1C40F', 
    'ALTO': COLORS['warning'],  'CRÍTICO': COLORS['danger'],
    'SIN_DATO': '#BDC3C7'
}

df_plot = df_2025.sort_values('ratio_ei', ascending=True)
colores = [nivel_color_map.get(n, '#BDC3C7') for n in df_plot['nivel_riesgo']]

fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(df_plot['departamento'], df_plot['ratio_ei'], color=colores, edgecolor='white', linewidth=0.5)

# Líneas de umbral
ax.axvline(x=1, color='gray', linestyle='-', linewidth=1.5, label='Equilibrio (1x)')
ax.axvline(x=4, color=COLORS['warning'], linestyle='--', linewidth=1.5, label='Umbral ALTO (4x)')
ax.axvline(x=6, color=COLORS['danger'],  linestyle='--', linewidth=1.5, label='Umbral CRÍTICO (6x)')

# Etiquetas de valor
for bar, val in zip(bars, df_plot['ratio_ei']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, 
            f'{val:.2f}x', va='center', fontsize=8.5)

# Leyenda de colores
parches = [mpatches.Patch(color=v, label=k) for k, v in nivel_color_map.items() if k != 'SIN_DATO']
ax.legend(handles=parches + [plt.Line2D([0],[0], color='gray', label='Equilibrio'),
          plt.Line2D([0],[0], color=COLORS['warning'], linestyle='--', label='Umbral ALTO'),
          plt.Line2D([0],[0], color=COLORS['danger'],  linestyle='--', label='Umbral CRÍTICO')],
          fontsize=8, loc='lower right')

ax.set_xlabel('Ratio Egresos / Ingresos', fontsize=11)
ax.set_title('Ratio Egresos/Ingresos por Departamento — IGSS 2025\n'
             'Valores > 1 indican que se gasta más de lo que se recauda', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/grafica_ratio_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada: exports/grafica_ratio_departamentos.png')

In [ ]:
# ── Gráfica 3: Scatter egresos vs ingresos ────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))

for nivel, grupo in df_2025.groupby('nivel_riesgo'):
    color = nivel_color_map.get(nivel, '#BDC3C7')
    ax.scatter(grupo['ingresos_q']/1e6, grupo['egresos_q']/1e6,
               c=color, s=100, label=nivel, alpha=0.85, edgecolors='white', linewidths=0.5)
    for _, row in grupo.iterrows():
        ax.annotate(row['departamento'],
                    xy=(row['ingresos_q']/1e6, row['egresos_q']/1e6),
                    xytext=(5, 3), textcoords='offset points', fontsize=7, alpha=0.9)

# Línea de equilibrio
max_val = max(df_2025['egresos_q'].max(), df_2025['ingresos_q'].max()) / 1e6
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.4, linewidth=1.5, label='Equilibrio E=I')

ax.set_xlabel('Ingresos (Q millones)', fontsize=11)
ax.set_ylabel('Egresos (Q millones)', fontsize=11)
ax.set_title('Egresos vs. Ingresos por Departamento — IGSS 2025\n'
             'Los puntos sobre la diagonal gastan más de lo que recaudan', 
             fontsize=12, fontweight='bold')
ax.legend(title='Nivel de riesgo', fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}M'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}M'))

plt.tight_layout()
plt.savefig('../exports/grafica_scatter_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada: exports/grafica_scatter_riesgo.png')

## 4. Análisis Estadístico — Prueba de Hipótesis

¿El incremento de costos en 2020-2021 es estadísticamente significativo?

In [ ]:
# Prueba t de Student: pre-pandemia vs pandemia
pre  = df_hosp[df_hosp['anio'] <= 2019]['costo_unitario_q'].values
pan  = df_hosp[df_hosp['anio'].isin([2020, 2021])]['costo_unitario_q'].values
post = df_hosp[df_hosp['anio'] >= 2022]['costo_unitario_q'].values

# Test Mann-Whitney (no paramétrico — muestras pequeñas)
stat1, p1 = stats.mannwhitneyu(pre, pan, alternative='less')
stat2, p2 = stats.mannwhitneyu(pre, post, alternative='less')

print('═' * 60)
print('PRUEBA MANN-WHITNEY — Significancia del incremento de costos')
print('═' * 60)
print(f'H₀: La distribución de costos pre-pandemia = pandemia')
print(f'H₁: Los costos en pandemia son mayores que pre-pandemia')
print(f'')
print(f'Pre-pandemia vs Pandemia:   U={stat1:.1f}, p={p1:.4f}')
print(f'  → {"RECHAZAR H₀ ✅" if p1 < 0.05 else "No se rechaza H₀"} (α=0.05)')
print(f'')
print(f'Pre-pandemia vs Post-pandemia: U={stat2:.1f}, p={p2:.4f}')
print(f'  → {"RECHAZAR H₀ ✅" if p2 < 0.05 else "No se rechaza H₀"} (α=0.05)')
print('═' * 60)

# Regresión lineal para tendencia
slope, intercept, r, p_reg, se = stats.linregress(df_hosp['anio'], df_hosp['costo_unitario_q'])
print(f'\nTENDENCIA (Regresión lineal):')
print(f'  Incremento anual promedio: Q{slope:,.2f}/año')
print(f'  R² = {r**2:.4f} (explicación del {r**2*100:.1f}% de la varianza)')
print(f'  p-value = {p_reg:.4f} → {"significativa" if p_reg < 0.05 else "no significativa"}')
print(f'  Proyección 2028: Q{slope*2028 + intercept:,.2f}')

## 5. Red Flags — Resumen de Hallazgos

In [ ]:
# Cargar red flags
df_flags = pd.read_sql_query("""
    SELECT rf.tipo_flag, rf.descripcion, rf.criticidad,
           d.nombre AS departamento, t.anio,
           rf.valor_detectado, rf.umbral
    FROM red_flags_log rf
    LEFT JOIN dim_departamento d ON rf.id_departamento = d.id_departamento
    LEFT JOIN dim_tiempo t ON rf.id_tiempo = t.id_tiempo
    ORDER BY rf.criticidad DESC, t.anio DESC
""", conn)

print('═' * 50)
print('RESUMEN DE RED FLAGS')
print('═' * 50)
print(df_flags.groupby(['criticidad', 'tipo_flag']).size().reset_index(name='count').to_string(index=False))
print('═' * 50)

# Mostrar alertas ALTA criticidad
alta = df_flags[df_flags['criticidad'] == 'ALTA']
if not alta.empty:
    print(f'\n🔴 {len(alta)} Red Flags de ALTA criticidad:')
    for _, row in alta.iterrows():
        print(f'  [{row["tipo_flag"]}] {row["departamento"]} ({row["anio"]}): {row["descripcion"][:80]}...')

In [ ]:
# Cerrar conexión
conn.close()
print('\n✅ Análisis exploratorio completado.')
print('   Las gráficas se guardaron en: exports/')
print('   Ejecuta el dashboard para visualización interactiva:')
print('   → python src/dashboard/app.py')